# Lab File Sorting and De-identifying

<a href="https://colab.research.google.com/github/PeaceAndLongLife/Analysis-Colab/blob/Development/notebooks/LabFilesSorting_and_de-identifying_Files.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# @title ## Mount google drive and read in the SERVICE_ACCOUNT_FILE  {"form-width":"20%"}

# @markdown ---
# @markdown
# @markdown The `SERVICE_ACCOUNT_FILE` path is stored as a secret in google colab. If you do not have this sotred on your colab, contact the Admin: Travis Kregear at tkregear@pdx.edu

# Mount Drive
from google.colab import drive
drive.mount('/content/drive')

# load SERVICE_ACCOUNT_FILE
from google.colab import userdata

SERVICE_ACCOUNT_FILE = userdata.get('SERVICE_ACCOUNT_FILE')


# @title ## Install pyDrive  {"form-width":"20%"}

# @markdown ---
# @markdown Installing PyDrive
# @markdown
# @markdown This is necessary if using the Google API to call files by their file_id

# !pip install PyDrive

import sys
# 2. Tell Python to look in your custom folder
sys.path.append('/content/drive/Shareddrives/AI_Shared/AI Colabs/packages')

# 3. Import your file!
from GoogleFunctions import GoogleDocumentManager, extract_file_id


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:

# @title Read in Userprofile and labfiles {"form-width":"20%"}
userprofile_link = "https://drive.google.com/file/d/1LMcLlQyvP15V3ApIj-9oG7P-K03WBbiU/view?usp=drive_link" # @param {"type":"string"}
userprofile_link_id = extract_file_id(userprofile_link)
csv_link = 'https://drive.google.com/file/d/1BMuOKccL7IOmj1KmtlnzKGyIwHjPneAd/view?usp=drive_link'# @param {"type":"string"}
show_data = False # @param {"type":"boolean"}
csv_link_id = extract_file_id(csv_link)
GDM = GoogleDocumentManager(SERVICE_ACCOUNT_FILE)

print(f"Reading in userprofile csv file. id={userprofile_link_id}")
userprofile_db = GDM.read_csv(
    document_id=userprofile_link_id,
    file_type = 'csv',
    )
if show_data:
  display(userprofile_db)

print(f"Reading in userprofile csv file. id={csv_link_id}")
csv_db = GDM.read_csv(
    document_id=csv_link_id,
    file_type = 'csv',
    )
if show_data:
  display(csv_db)

In [ ]:
# @title ## Remove Staff users {"form-width":"20%"}
show_remove_staff_data = False # @param {"type":"boolean"}

# @markdown Remove any users listed as Staff so the data only includes student data.
# @markdown
# @markdown  Additional users removed:
# @markdown         'travis@pdx.edu',
# @markdown         'marie.snetinova@matfyz.cuni.cz',
# @markdown         'mptl_group_1@pdx.edu',
# @markdown         'mptl_group_2@pdx.edu',
# @markdown         'mptl_group_3@pdx.edu',
# @markdown         'mptl_group_4@pdx.edu',
# @markdown         'mptl_group_5@pdx.edu',
# @markdown         'mptl_group_6@pdx.edu',
# @markdown         'mptl_user_7@pdx.edu',
# @markdown         'mptl_group_1@pdx.edu',
# @markdown         'mptl_user_1@pdx.edu',
# @markdown         'mptl_user_10@pdx.edu'


# Generate a list of users where is_staff is TRUE from userprofile
if 'is_staff' in userprofile_db.columns:
    staff_users_df = userprofile_db[userprofile_db['is_staff'] == True]

    if 'username' in staff_users_df.columns:
      staff_user_ids = staff_users_df['username'].dropna().astype(str).tolist()
      staff_user_ids = staff_user_ids + [
          'travis@pdx.edu',
          'marie.snetinova@matfyz.cuni.cz',
          'mptl_group_1@pdx.edu',
          'mptl_group_2@pdx.edu',
          'mptl_group_3@pdx.edu',
          'mptl_group_4@pdx.edu',
          'mptl_group_5@pdx.edu',
          'mptl_group_6@pdx.edu',
          'mptl_user_7@pdx.edu',
          'mptl_group_1@pdx.edu',
          'mptl_user_1@pdx.edu',
          'mptl_user_10@pdx.edu'
      ]

      print(f"Found {len(staff_user_ids)} staff users. IDs: {staff_user_ids}")
      # display(staff_users_df)

      # Remove any row from files from users on that list


      # display(current_df.head())
      # if 'username' in csv_db.columns:
      #     csv_db = csv_db.rename(columns={'username': 'user'})
      #     print(f"Renaming 'username' column to 'user' in csv_db")

      if 'user' in csv_db.columns:
          initial_chat_rows = len(csv_db)
          no_staff_df = csv_db[~csv_db['user'].isin(staff_user_ids)]
          removed_chat_rows = initial_chat_rows - len(no_staff_df)
          csv_db = no_staff_df
          print(f"Removed {removed_chat_rows} rows from csv_db corresponding to staff users.")
          print(f"Updated csv_db DataFrame after removing staff user entries.")

          if show_remove_staff_data:
            display(no_staff_df)
      else:
          print(f"Warning: 'user' column not found in csv_db DataFrame. Cannot remove staff user entries.")

else:
    print(f"Warning: 'is_staff' column not found in the DataFrame. Cannot identify staff users.")


In [4]:
# @title Iterate through labfile, report on Parent ID status {"form-width":"20%"}
check_file = False # @param {"type":"boolean"}
expected_parent_folder_link = "https://drive.google.com/drive/folders/1zSBgR9rkGrkFlR_kqSlqd_D22wB-wtHL?usp=drive_link" # @param {"type":"string"}

# # Extract file IDs from URLs if provided
expected_parent_folder_id = extract_file_id(expected_parent_folder_link)

GDM = GoogleDocumentManager(SERVICE_ACCOUNT_FILE)

import pandas as pd
from tqdm.notebook import tqdm

tqdm.pandas()

def _check_parent_folder(file_id):
    # Helper function to check parent directory for a given file_id
    props = GDM.get_file_parent_info(
        file_id=file_id,
    )
    if type(props) == str:
      return 'missing'
    elif props['parents'][0]['id'] == expected_parent_folder_id:
      return True
    else:
      return False

def check_file_function(row):
    doc_dir = _check_parent_folder(row['doc_id'])
    sheet_dir = _check_parent_folder(row['sheet_id'])
    return doc_dir, sheet_dir



if check_file:
  csv_db[['Correct DOC', 'Correct SHEET']] = csv_db.progress_apply(
      check_file_function,
      axis=1,
      result_type='expand'
  )
  display(f"Number of rows where Correct DOC is False: {len(csv_db[csv_db['Correct DOC'] == False][["Correct DOC", "Correct SHEET", "id"]])}")
  display(csv_db[csv_db['Correct DOC'] == False][["Correct DOC", "Correct SHEET", "id"]])
  display(csv_db[csv_db['Correct SHEET'] == False][["Correct DOC", "Correct SHEET", "id"]])

Google Drive API service built successfully.


In [ ]:
# @title Iterate through labfile, rename and copy Document to new folder with user_id inplace of username{"form-width":"20%"}
check_file = False # @param {"type":"boolean"}
show_result = False # @param {"type":"boolean"}
destination_folder_link = "https://drive.google.com/drive/folders/1e8WHd-W3OobjeUW-_wSxOQLR4gojBMzn?usp=drive_link" # @param {"type":"string"}
destination_folder_id = extract_file_id(destination_folder_link)

# @markdown ---

write_csv = False # @param {"type":"boolean"}
csv_folder_link = "https://drive.google.com/drive/folders/18bnV8u9CIf-G09BGDjpPhwFwrAd1c5Hj?usp=drive_link" # @param {"type":"string"}
csv_folder_id = extract_file_id(destination_folder_link)

# @markdown ---
# @markdown This cell will timeout with an error, ""WARNING:googleapiclient.http:Encountered 403 Forbidden with reason "userRateLimitExceeded" after 1031/1112. to avoid I have added a time.sleep(0.5) for the code to pause every cycle for 500ms) However the data copied appears to be complete. I might need to add a cell to check the data that is copied and identify missed files, then copy them specifically.


GDM = GoogleDocumentManager(SERVICE_ACCOUNT_FILE)

import time
import pandas as pd
from tqdm.notebook import tqdm

tqdm.pandas()

def copy_rename_file(file_id, user_id, assignment_name, folder_link_id):
  # Helper function to check parent directory for a given file_id

  # # # commented for file integrity. uncomment to run.
  # new_file_id = GDM.copy_file(
  #     file_id = file_id,
  #     new_name = f"{user_id} - {assignment_name}",
  #     parent_folder_id = folder_link_id,
  # )

  new_file_id = file_id
  return new_file_id

def check_file_function(row):
      deident_id = userprofile_db[userprofile_db['username'] == row['user']]['id'].iloc[0]
    # if deident_id == 103:
      time.sleep(0.5)
      processed_doc_id = copy_rename_file(
            file_id = row['doc_id'],
            user_id = deident_id,
            assignment_name = row['assignment'],
            folder_link_id = destination_folder_id,
            )
      processed_sheet_id = copy_rename_file(
            file_id = row['sheet_id'],
            user_id = deident_id,
            assignment_name = row['assignment'],
            folder_link_id = destination_folder_id,
            )
      return pd.Series({
          "user": deident_id,
          # "sheet_id": row['sheet_id'],
          # "doc_id": row['doc_id'],
          "sheet_id": processed_sheet_id,
          "doc_id": processed_doc_id,
          "Assignment": row['assignment'],
        })
    # return pd.Series({
    #     "user": row['user'],
    #     "sheet_id": row['sheet_id'],
    #     "doc_id": row['doc_id'],
    #     # "sheet_id": processed_sheet_id,
    #     # "doc_id": processed_doc_id,
    #     "Assignment": row['assignment'],
    #   })




if check_file:
  new_labfiles_df = csv_db.progress_apply(
      check_file_function,
      axis=1,
  )

  if show_result:
    display(new_labfiles_df)

  if write_csv:
    csv_folder_id = extract_file_id(csv_folder_link)
    uploaded_file_id = GDM.upload_dataframe_to_drive(new_labfiles_df, "lab_files_de-identified.csv", csv_folder_id)
    print(f"Sucessfully Uploaded file_id: {uploaded_file_id} to target folder {csv_folder_id}")


Google Drive API service built successfully.


  0%|          | 0/1112 [00:00<?, ?it/s]

In [34]:
display(copied_new_labfiles_df)

,user,sheet_id,doc_id,Assignment
0,146,1zHjvgdMPdCsVz5ysRtz01rjroiH5pZf1iuvOx_7tigw,1ZkZR3B-HjyzLKlTuaTNb1MRL5kj2vD1NmsYMJU6Abh4,PH214 (online) - Assignment 8
1,146,1or11iZqXVzpFG3XVtSOfzYj94BBgSQXT4CEcajAeMQo,1vWa6zagWncBZjDV2JstIwY5A_74EkE3VjIIAD9lfANM,PH214 (online) - Assignment 6
2,159,1NcAvLpYIXCAARzQGqDNESipOttc40BIHa20_WouO-iw,1T8cL01ZOYUtjFJjG8akmfhWJ9s2PkPdGk3E3E7_3POk,PH214 (online) - Assignment 8
3,267,1RfYMbRtxuCBdy719usW5R5rqA1c_7KVOPpRQUgq45Y0,1nCM7fxgU5XBDl1_OKrEp9p1CMroxpQ-MTIvPzjfeK5Y,PH214 (online) - Assignment 6
4,146,1jca2CsabrK6y46oHO3nCQKUMjSooQZXbmOVPpa7tI_k,1XRAnRgGC4lQM9KtJSCIESkMSzNhW_GCqM097V1g8v5w,PH214 (online) - Assignment 7
...,...,...,...,...
1107,267,1Gpz7lAs15gwxhf87A6sYMiC4Lx9iSJAhzs_DziKLMnM,16GU1yEfmI_C3teFn0p_6McrhAwpUyGbIQOiNSc24uS4,PH214 (online) - Assignment 4
1108,267,1LrSQSbESyzeUh4UYSZMXTAzAqSYBoRriQmjf0ymyZiM,1-l4ygTJb8tdGf1Uhq46-mDe6pv0A4xURWHhwThgAKb4,PH214 (online) - Assignment 5
1109,124,1ZuKi0QBZ8WDE-vnFiiiyIfU5jAFwVCUN9MPrZEfjljw,11-JwJMdr48XCSgG05uQgPdYqplXvuBRSKQ0evQi8P0E,PH214 (online) - Assignment 7
1110,267,1mfRV1-XewjDY5HfrDDoN4LS3ROypjeIWNITgpmDP0FU,150BWr8gpsV8AWd0EyB4C79OnTw-3FYz8zJiXKJxVx-U,PH214 (online) - Assignment 7


In [36]:
from google.colab import sheets

# @title Create Interactive Sheet and optionally move to a specific folder {"form-width":"20%"}
move_to_target_folder = False # @param {type:"boolean"}
target_folder_for_sheet_link = "https://drive.google.com/drive/folders/18bnV8u9CIf-G09BGDjpPhwFwrAd1c5Hj?usp=drive_link" # @param {type:"string"}

# Create the interactive sheet
sheet_object = sheets.InteractiveSheet(title="lab_files_de-ident_sheet", include_column_headers=True, df=copied_new_labfiles_df)

# Get the ID of the newly created sheet
sheet_id = sheet_object.id
print(f"Interactive Sheet created with ID: {sheet_id}")
print(f"Interactive Sheet URL: {sheet_object.url}")

if move_to_target_folder:
  if 'GDM' not in globals():
    GDM = GoogleDocumentManager(SERVICE_ACCOUNT_FILE)

  target_folder_id = extract_file_id(target_folder_for_sheet_link)
  if target_folder_id:
    try:
      # Get current parent(s) to remove from
      current_parents = GDM.get_file_parent_info(file_id=sheet_id)['parents']
      current_parent_ids = [p['id'] for p in current_parents]

      # Move the file by updating its parents
      GDM.update_file_metadata(
          file_id=sheet_id,
          add_parents=target_folder_id,
          remove_parents=','.join(current_parent_ids)
      )
      print(f"Successfully moved Interactive Sheet (ID: {sheet_id}) to folder (ID: {target_folder_id})")
    except Exception as e:
      print(f"Error moving sheet to target folder: {e}")
      print("Please ensure the target folder link is valid and you have appropriate permissions.")
  else:
    print(f"Invalid target folder link provided: {target_folder_for_sheet_link}")


https://docs.google.com/spreadsheets/d/10AaQ6bci69Fvff7f4TTljzhuR-VgvtjPp1bV4NvEJgw/edit#gid=0
